In [1]:
import polars as pl
from project_paths  import paths

In [2]:

curves = {
    # embankment, varying core (pg. 75 fluvial, pg. 77 coastal), narrow
    ("embankment", "fluvial", "varying_core", "narrow"): {
        ("slowest", 1): [0, 5, 10, 40, 60], 
        ("slowest", 2): [0, 20, 40, 70, 110], 
        ("slowest", 3): [0, 22, 44, 90, 130],
        ("medium", 1): [0, 3, 6, 25, 40],   
        ("medium", 2): [0, 15, 30, 60, 80],   
        ("medium", 3): [0, 16, 33, 70, 90],
        ("fastest", 1): [0, 1, 3, 5, 7],    
        ("fastest", 2): [0, 2, 5, 7, 10],     
        ("fastest", 3): [0, 3, 6, 8, 11],
    },
    ("embankment", "coastal", "varying_core", "narrow"): {
        ("slowest", 1): [0, 5, 10, 40, 60],
        ("slowest", 2): [0, 20, 40, 60, 80],
        ("slowest", 3): [0, 22, 45, 80, 110],
        ("medium", 1): [0, 3, 6, 22, 30],
        ("medium", 2): [0, 14, 28, 40, 50],
        ("medium", 3): [0, 15, 30, 45, 60],
        ("fastest", 1): [0, 1, 2, 4, 5],
        ("fastest", 2): [0, 2, 4, 6, 8],
        ("fastest", 3): [0, 3, 5, 8, 10],
    },

    ("embankment", "coastal", "varying_core", "wide"): {
        ("slowest", 1): [0, 5, 10, 40, 60],
        ("slowest", 2): [0, 20, 40, 70, 90],
        ("slowest", 3): [0, 22, 44, 85, 120],
        ("medium", 1): [0, 4, 6, 22, 30],
        ("medium", 2): [0, 14, 30, 50, 60],
        ("medium", 3): [0, 20, 35, 55, 70],
        ("fastest", 1): [0, 2, 5, 9, 12],
        ("fastest", 2): [0, 4, 9, 12, 18],
        ("fastest", 3): [0, 5, 10, 14, 20],
    },
    ("embankment", "fluvial", "varying_core", "wide"): {
        ("slowest", 1): [0, 5, 10, 40, 60],
        ("slowest", 2): [0, 20, 40, 70, 110],
        ("slowest", 3): [0, 22, 44, 90, 130],
        ("medium", 1): [0, 3, 6, 25, 40],
        ("medium", 2): [0, 15, 30, 60, 80],
        ("medium", 3): [0, 16, 33, 70, 90],
        ("fastest", 1): [0, 2, 6, 10, 14],
        ("fastest", 2): [0, 4, 10, 14, 20],
        ("fastest", 3): [0, 5, 10, 14, 20],
    },

    # embankment with toe protection or reventment pg. 82

    ("protected_embankment", "fluvial", "earth", "narrow"): {
        ("slowest", 1): [0, 20, 40, 60, 80],
        ("slowest", 2): [0, 25, 50, 80, 130],
        ("slowest", 3): [0, 30, 60, 90, 140],
        ("medium", 1): [0, 15, 25, 35, 40],
        ("medium", 2): [0, 20, 30, 70, 90],
        ("medium", 3): [0, 25, 45, 80, 100],
        ("fastest", 1): [0, 3, 8, 10, 12],
        ("fastest", 2): [0, 3, 8, 10, 15],
        ("fastest", 3): [0, 15, 20, 30, 40],
    },
    ("protected_embankment", "fluvial", "earth", "wide"): {
        ("slowest", 1): [0, 20, 40, 60, 80],
        ("slowest", 2): [0, 25, 50, 100, 130],
        ("slowest", 3): [0, 30, 60, 110, 150],
        ("medium", 1): [0, 15, 25, 35, 40],
        ("medium", 2): [0, 20, 30, 70, 90],
        ("medium", 3): [0, 25, 45, 80, 110],
        ("fastest", 1): [0, 8, 15, 20, 25],
        ("fastest", 2): [0, 12, 20, 30, 40],
        ("fastest", 3): [0, 15, 30, 40, 50],
    },
    ("protected_embankment", "coastal", "earth", "narrow"): {
        ("slowest", 1): [0, 10, 20, 40, 60],
        ("slowest", 2): [0, 20, 50, 75, 100],
        ("slowest", 3): [0, 30, 60, 100, 130],
        ("medium", 1): [0, 9, 19, 31, 40],
        ("medium", 2): [0, 15, 30, 50, 60],
        ("medium", 3): [0, 20, 40, 60, 80],
        ("fastest", 1): [0, 3, 7, 10, 12],
        ("fastest", 2): [0, 3, 8, 10, 15],
        ("fastest", 3): [0, 10, 20, 25, 30],
    },
    ("protected_embankment", "coastal", "earth", "wide"): {
        ("slowest", 1): [0, 20, 40, 60, 80],
        ("slowest", 2): [0, 25, 50, 90, 120],
        ("slowest", 3): [0, 30, 60, 100, 140],
        ("medium", 1): [0, 9, 19, 31, 40],
        ("medium", 2): [0, 15, 30, 50, 60],
        ("medium", 3): [0, 20, 40, 60, 80],
        ("fastest", 1): [0, 8, 15, 20, 25],
        ("fastest", 2): [0, 12, 20, 30, 40],
        ("fastest", 3): [0, 15, 30, 40, 50],
    },

    # vertical wall, concrete/brick-masonry (from Table 2.3)
    ("vertical_wall", "fluvial", "concrete_masonry", "na"): {
        ("slowest", 1): [0, 20, 50, 70, 80],
        ("slowest", 2): [0, 25, 60, 100, 120],
        ("slowest", 3): [0, 30, 70, 130, 160],
        ("medium", 1): [0, 15, 35, 50, 60],
        ("medium", 2): [0, 20, 45, 70, 90],
        ("medium", 3): [0, 25, 55, 90, 120],
        ("fastest", 1): [0, 5, 20, 30, 40],
        ("fastest", 2): [0, 10, 30, 50, 60],
        ("fastest", 3): [0, 15, 40, 70, 80],
    },
    ("vertical_wall", "coastal", "concrete_masonry", "na"): {
        ("slowest", 1): [0, 15, 45, 60, 80], 
        ("slowest", 2): [0, 20, 60, 80, 100], 
        ("slowest", 3): [0, 25, 75, 100, 120],
        ("medium", 1): [0, 10, 30, 40, 50],  
        ("medium", 2): [0, 15, 40, 55, 70],   
        ("medium", 3): [0, 20, 50, 70, 90],
        ("fastest", 1): [0, 5, 15, 25, 30],  
        ("fastest", 2): [0, 10, 20, 30, 40],  
        ("fastest", 3): [0, 15, 25, 35, 50],
    },

    # vertical wall, timber (from pg. 48,49) 
    ("vertical_wall", "fluvial", "timber", "na"): {
        ("slowest", 1): [0, 7, 15, 18, 21],
        ("slowest", 2): [0, 15, 30, 35, 40],
        ("slowest", 3): [0, 23, 45, 52, 60],
        ("medium", 1): [0, 5, 10, 12, 15],
        ("medium", 2): [0, 10, 20, 25, 30],
        ("medium", 3): [0, 15, 30, 35, 42],
        ("fastest", 1): [0, 3, 5, 7, 10],
        ("fastest", 2): [0, 5, 10, 12, 15],
        ("fastest", 3): [0, 7, 15, 17, 20],
    },
    ("vertical_wall", "coastal", "timber", "na"): {
        ("slowest", 1): [0, 5, 13, 16, 20], 
        ("slowest", 2): [0, 14, 28, 33, 38], 
        ("slowest", 3): [0, 21, 42, 48, 55],
        ("medium", 1): [0, 4, 8, 10, 14],   
        ("medium", 2): [0, 8, 18, 23, 28],   
        ("medium", 3): [0, 13, 28, 33, 38],
        ("fastest", 1): [0, 2, 4, 6, 8],    
        ("fastest", 2): [0, 4, 8, 10, 13],   
        ("fastest", 3): [0, 5, 13, 15, 18],
    },

    # flood gates and barriers, metal (from pg. 196 197)
    ("flood_gate", "fluvial", "metal", "na"): {
        ("slowest", 1): [0, 15, 32, 41, 50], 
        ("slowest", 2): [0, 20, 40, 50, 60], 
        ("slowest", 3): [0, 25, 48, 59, 70],
        ("medium", 1): [0, 12, 25, 32, 38],  
        ("medium", 2): [0, 18, 34, 42, 50],  
        ("medium", 3): [0, 24, 43, 52, 62],
        ("fastest", 1): [0, 5, 12, 16, 20],  
        ("fastest", 2): [0, 10, 22, 30, 35], 
        ("fastest", 3): [0, 15, 32, 44, 50],
    },
    ("flood_gate", "coastal", "metal", "na"): {
        ("slowest", 1): [0, 13, 22, 26, 30], 
        ("slowest", 2): [0, 18, 29, 35, 40], 
        ("slowest", 3): [0, 23, 36, 44, 50],
        ("medium", 1): [0, 10, 14, 16, 18],  
        ("medium", 2): [0, 15, 23, 27, 30],  
        ("medium", 3): [0, 20, 32, 38, 42],
        ("fastest", 1): [0, 4, 7, 9, 10],    
        ("fastest", 2): [0, 7, 11, 13, 15],  
        ("fastest", 3): [0, 10, 15, 17, 20],
    },


    # demountable defence (on pg. 67, tbl C.4 metal/wood)
    ("demountable_defence", "fluvial", "metal", "na"): {
        ("slowest", 1): [0, 2, 4, 5, 7],
        ("slowest", 2): [0, 10, 20, 60, 70],
        ("slowest", 3): [0, 15, 25, 70, 80],
        ("medium", 1): [0, 1, 3, 4, 5],
        ("medium", 2): [0, 5, 10, 45, 55],
        ("medium", 3): [0, 8, 15, 55, 65],
        ("fastest", 1): [0, 1, 2, 3, 4],
        ("fastest", 2): [0, 2, 5, 35, 45],
        ("fastest", 3): [0, 5, 10, 45, 55],
    },
    ("demountable_defence", "fluvial", "wood", "na"): {
        ("slowest", 1): [0, 2, 4, 5, 7],
        ("slowest", 2): [0, 5, 10, 30, 35],
        ("slowest", 3): [0, 8, 13, 35, 40],
        ("medium", 1): [0, 1, 3, 4, 5],
        ("medium", 2): [0, 3, 5, 23, 28],
        ("medium", 3): [0, 4, 8, 28, 33],
        ("fastest", 1): [0, 1, 2, 3, 4],
        ("fastest", 2): [0, 1, 3, 18, 23],
        ("fastest", 3): [0, 3, 5, 23, 28],
    },

    #  beach (C.7 pg 130)
    ("beach", "coastal", "shingle/sand", "na"): {
        ("slowest", 1): [0, 15, 38, 75, 100],
        ("slowest", 2): [0, 27, 50, 150, 200],
        ("slowest", 3): [0, 27, 75, 200, 250],
        ("medium", 1): [0, 9, 13, 25, 35],
        ("medium", 2): [0, 16, 30, 50, 75 ],
        ("medium", 3): [0, 20, 55, 90, 120],
        ("highest", 1): [0, 4, 7, 9, 13],
        ("highest", 2): [0, 7, 10, 13, 20],
        ("highest", 3): [0, 12, 20, 25, 40],
    },

    # dunes (C.15 pg 152)
    ("dunes", "coastal", "", ""): {
        ("slowest", 1): [0, 20, 40, 110, 150],
        ("slowest", 2): [0, 27, 60, 150, 200],
        ("slowest", 3): [0, 30, 80, 190, 250],
        ("medium", 1): [0, 10, 15, 30, 40],
        ("medium", 2): [0, 15, 35, 60, 80],
        ("medium", 3): [0,20, 60, 100, 130],
        ("highest", 1): [0, 5, 8, 10, 15],
        ("highest", 2): [0, 7, 10, 13, 20],
        ("highest", 3): [0, 12, 20, 25, 40],
    },

    # weir (C.18 pg. 167)
    ("weir", "fluvial", "all", "na"): {
        ("slowest", 1): [0, 20, 30, 50, 70],
        ("slowest", 2): [0, 40, 70, 90, 110],
        ("slowest", 3): [0, 60, 110, 130, 150],
        ("medium", 1): [0, 15, 20, 40, 60],
        ("medium", 2): [0, 30, 50, 70, 90],
        ("medium", 3): [0, 45, 80, 100, 120],
        ("highest", 1): [0, 10, 15, 30, 40],
        ("highest", 2): [0, 20, 30, 50, 60],
        ("highest", 3): [0, 30, 45, 70, 80],
    },

}


rows = []
for (htype, env, mat, width), grid in curves.items():

    for (rate, regime), t in grid.items():
        rows.append(
            {
                "halcrow_type": htype, 
                "environment": env, 
                "material": mat, 
                "width": width,
                "rate": rate, 
                "regime": regime,
                "t1": t[0],
                "t2": t[1],
                "t3": t[2],
                "t4": t[3],
                "t5": t[4],
            }
        )



curve_df = pl.DataFrame(rows)

curve_df

halcrow_type,environment,material,width,rate,regime,t1,t2,t3,t4,t5
str,str,str,str,str,i64,i64,i64,i64,i64,i64
"""embankment""","""fluvial""","""varying_core""","""narrow""","""slowest""",1,0,5,10,40,60
"""embankment""","""fluvial""","""varying_core""","""narrow""","""slowest""",2,0,20,40,70,110
"""embankment""","""fluvial""","""varying_core""","""narrow""","""slowest""",3,0,22,44,90,130
"""embankment""","""fluvial""","""varying_core""","""narrow""","""medium""",1,0,3,6,25,40
"""embankment""","""fluvial""","""varying_core""","""narrow""","""medium""",2,0,15,30,60,80
…,…,…,…,…,…,…,…,…,…,…
"""weir""","""fluvial""","""all""","""na""","""medium""",2,0,30,50,70,90
"""weir""","""fluvial""","""all""","""na""","""medium""",3,0,45,80,100,120
"""weir""","""fluvial""","""all""","""na""","""highest""",1,0,10,15,30,40


In [6]:
manual_data = paths.data / "individual_asset_investigations" / "assessed_assets.csv"

assets_df = pl.read_csv(manual_data)

assets_df = assets_df.filter(pl.col("usable")).select(
    "asset_id",
    pl.col("asset_type").alias("halcrow_type"),
    "material",
    "environment",
    pl.when(pl.col("width").eq("n/a")).then(pl.lit("na")).otherwise(pl.col("width")).alias("width"),
    "rate",
    "regime",
    pl.col("age").alias("age_years")
)

assets_df

asset_id,halcrow_type,material,environment,width,rate,regime,age_years
i64,str,str,str,str,str,i64,f64
327169,"""flood_gate""","""metal""","""fluvial""","""na""",null,2,31.304586
328183,"""flood_gate""","""metal""","""fluvial""","""na""",null,2,31.304586
328184,"""flood_gate""","""metal""","""fluvial""","""na""",null,2,31.304586
328185,"""flood_gate""","""metal""","""fluvial""","""na""",null,2,31.304586
328190,"""flood_gate""","""metal""","""fluvial""","""na""",null,2,31.304586
…,…,…,…,…,…,…,…
102898,"""embankment""","""varying_core""","""coastal""","""wide""",null,null,30.308008
105361,"""embankment""","""varying_core""","""coastal""","""wide""",null,null,35.312799
126419,"""protected_embankment""","""earth""","""coastal""","""wide""",null,null,36.13963


In [7]:
# to start with, using same rate and regime assumptions as before
# i think the regime assumption is really defensible, since all these assets are maintained by EA and target grades are 3
# rate being medium is fine as a start but then should do some sensitivity testing later


rate, regime = "medium", 2


def frac(low, high):
    low_e, high_e = pl.col(low), pl.col(high)
    return pl.when(high_e > low_e).then((pl.col("age_years") - low_e) / (high_e - low_e)).otherwise(0.0)




In [8]:
prediction = (
    assets_df
    .join(
        curve_df.filter((pl.col("rate").eq(rate)) & (pl.col("regime").eq(regime))),
        on=["halcrow_type", "environment", "material", "width"], 
        how="left",
    )
    .with_columns(
        pl.when(pl.col("t2").is_null()).then(pl.lit("unimplemented"))
            .otherwise(pl.lit("scored")).alias("curve_status")
    )
    .with_columns(
        pl.when(pl.col("age_years").ge(pl.col("t5"))).then(5.0)
            .when(pl.col("age_years").ge(pl.col("t4"))).then(4.0 + frac("t4", "t5"))
            .when(pl.col("age_years").ge(pl.col("t3"))).then(3.0 + frac("t3", "t4"))
            .when(pl.col("age_years").ge(pl.col("t2"))).then(2.0 + frac("t2", "t3"))
            .otherwise(1.0 + frac("t1", "t2"))
        .clip(1.0, 5.0)
        .alias("pred_grade_cont")
    )
    .with_columns(pl.col("pred_grade_cont").floor().cast(pl.Int8).clip(1, 5).alias("pred_grade"))
)


prediction


asset_id,halcrow_type,material,environment,width,rate,regime,age_years,rate_right,regime_right,t1,t2,t3,t4,t5,curve_status,pred_grade_cont,pred_grade
i64,str,str,str,str,str,i64,f64,str,i64,i64,i64,i64,i64,i64,str,f64,i8
327169,"""flood_gate""","""metal""","""fluvial""","""na""",null,2,31.304586,"""medium""",2,0,18,34,42,50,"""scored""",2.831537,2
328183,"""flood_gate""","""metal""","""fluvial""","""na""",null,2,31.304586,"""medium""",2,0,18,34,42,50,"""scored""",2.831537,2
328184,"""flood_gate""","""metal""","""fluvial""","""na""",null,2,31.304586,"""medium""",2,0,18,34,42,50,"""scored""",2.831537,2
328185,"""flood_gate""","""metal""","""fluvial""","""na""",null,2,31.304586,"""medium""",2,0,18,34,42,50,"""scored""",2.831537,2
328190,"""flood_gate""","""metal""","""fluvial""","""na""",null,2,31.304586,"""medium""",2,0,18,34,42,50,"""scored""",2.831537,2
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
102898,"""embankment""","""varying_core""","""coastal""","""wide""",null,null,30.308008,"""medium""",2,0,14,30,50,60,"""scored""",3.0154,3
105361,"""embankment""","""varying_core""","""coastal""","""wide""",null,null,35.312799,"""medium""",2,0,14,30,50,60,"""scored""",3.26564,3
126419,"""protected_embankment""","""earth""","""coastal""","""wide""",null,null,36.13963,"""medium""",2,0,15,30,50,60,"""scored""",3.306982,3


In [9]:
actual_grade_data = paths.processed_data / "unified_aims_eir_bgs.parquet"

actual_grades = pl.read_parquet(actual_grade_data).select(
    pl.col("aims__asset_id").alias("asset_id"),
    pl.col("eir__condition_grade").alias("actual_grade")
)

In [10]:
comp = prediction.join(other=actual_grades, how="left", on="asset_id")

comp

asset_id,halcrow_type,material,environment,width,rate,regime,age_years,rate_right,regime_right,t1,t2,t3,t4,t5,curve_status,pred_grade_cont,pred_grade,actual_grade
i64,str,str,str,str,str,i64,f64,str,i64,i64,i64,i64,i64,i64,str,f64,i8,f64
327169,"""flood_gate""","""metal""","""fluvial""","""na""",null,2,31.304586,"""medium""",2,0,18,34,42,50,"""scored""",2.831537,2,2.0
328183,"""flood_gate""","""metal""","""fluvial""","""na""",null,2,31.304586,"""medium""",2,0,18,34,42,50,"""scored""",2.831537,2,2.0
328184,"""flood_gate""","""metal""","""fluvial""","""na""",null,2,31.304586,"""medium""",2,0,18,34,42,50,"""scored""",2.831537,2,2.0
328185,"""flood_gate""","""metal""","""fluvial""","""na""",null,2,31.304586,"""medium""",2,0,18,34,42,50,"""scored""",2.831537,2,2.0
328190,"""flood_gate""","""metal""","""fluvial""","""na""",null,2,31.304586,"""medium""",2,0,18,34,42,50,"""scored""",2.831537,2,2.0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
102898,"""embankment""","""varying_core""","""coastal""","""wide""",null,null,30.308008,"""medium""",2,0,14,30,50,60,"""scored""",3.0154,3,3.0
105361,"""embankment""","""varying_core""","""coastal""","""wide""",null,null,35.312799,"""medium""",2,0,14,30,50,60,"""scored""",3.26564,3,3.0
126419,"""protected_embankment""","""earth""","""coastal""","""wide""",null,null,36.13963,"""medium""",2,0,15,30,50,60,"""scored""",3.306982,3,2.0


In [12]:
with pl.Config(tbl_rows=-1):
    print(comp.select("asset_id", "pred_grade", "actual_grade"))

shape: (11, 3)
┌──────────┬────────────┬──────────────┐
│ asset_id ┆ pred_grade ┆ actual_grade │
│ ---      ┆ ---        ┆ ---          │
│ i64      ┆ i8         ┆ f64          │
╞══════════╪════════════╪══════════════╡
│ 327169   ┆ 2          ┆ 2.0          │
│ 328183   ┆ 2          ┆ 2.0          │
│ 328184   ┆ 2          ┆ 2.0          │
│ 328185   ┆ 2          ┆ 2.0          │
│ 328190   ┆ 2          ┆ 2.0          │
│ 328191   ┆ 2          ┆ 2.0          │
│ 102898   ┆ 3          ┆ 3.0          │
│ 105361   ┆ 3          ┆ 3.0          │
│ 126419   ┆ 3          ┆ 2.0          │
│ 133876   ┆ 3          ┆ 3.0          │
│ 134914   ┆ 5          ┆ 3.0          │
└──────────┴────────────┴──────────────┘


In [16]:
comparison_grade = comp.select(
    (pl.col("pred_grade") - pl.col("actual_grade")).abs().mean().alias("mae"),
)

comparison_grade_cont = comp.select(
    (pl.col("pred_grade_cont") - pl.col("actual_grade")).abs().mean().alias("mae_cont"),
)

print(comparison_grade, "\n\n", comparison_grade_cont)

shape: (1, 1)
┌──────────┐
│ mae      │
│ ---      │
│ f64      │
╞══════════╡
│ 0.272727 │
└──────────┘ 

 shape: (1, 1)
┌──────────┐
│ mae_cont │
│ ---      │
│ f64      │
╞══════════╡
│ 0.812599 │
└──────────┘


In [20]:
import numpy as np

heuristic_vs_baselines = comp.select(
    pl.col("actual_grade"),
    pl.col("pred_grade")
).with_columns(
    pl.lit(value=3).alias("flat_baseline"),
    pl.Series(values=[np.random.randint(low=1, high=6) for _ in range(comp.height)]).alias("random_baseline")
)

heuristic_vs_baselines = heuristic_vs_baselines.select(
    (pl.col("pred_grade") - pl.col("actual_grade")).abs().mean().alias("mae"),
    (pl.col("flat_baseline") - pl.col("actual_grade")).abs().mean().alias("flt_base_mae"),
    (pl.col("random_baseline") - pl.col("actual_grade")).abs().mean().alias("rand_base_mae"),
)

heuristic_vs_baselines

mae,flt_base_mae,rand_base_mae
f64,f64,f64
0.272727,0.636364,1.272727


In [23]:
import numpy as np

heuristic_vs_baselines = comp.select(
    pl.col("asset_id"),
    pl.col("halcrow_type"),
    pl.col("actual_grade"),
    pl.col("pred_grade")
).with_columns(
    pl.lit(value=3).alias("flat_baseline"),
    pl.Series(values=[np.random.randint(low=1, high=6) for _ in range(comp.height)]).alias("random_baseline")
).filter(pl.col("halcrow_type").is_in(["embankment", "protected_embankment"]))

heuristic_vs_baselines = heuristic_vs_baselines.select(
    (pl.col("pred_grade") - pl.col("actual_grade")).abs().mean().alias("mae"),
    (pl.col("flat_baseline") - pl.col("actual_grade")).abs().mean().alias("flt_base_mae"),
    (pl.col("random_baseline") - pl.col("actual_grade")).abs().mean().alias("rand_base_mae"),
)

heuristic_vs_baselines

mae,flt_base_mae,rand_base_mae
f64,f64,f64
0.6,0.2,0.8


The gates were all grade 2, so the "median" (3) obviously performs badly. There were also the ones where the heuristic performed well, and also where all of my previous assumptions held.

The embankments are the more interesting case. They were the ones that took more work to collect:

    1. extract centroid coordinates and locate on google maps
    2. inspect aerial satallite imagery to confirm environment and identify material and asset type (is protection or revampment visible?)
    3. find gis data on elevation (and high quality aerial photography if possible). Load data, layer aims defence gis over map imagery and lidar composite data to identify flood defence assets, and find the elevation profile across the defence to identify narrow vs wide assets.

The assumptions made in notebook 8 did not hold for these - one asset changed from embankment to protected_embankment due do a visible concrete footing, and there was a mix of narrow and wide crests. 

The mae of the grade of these embankments is much better than in notebook 8 now that these assumptions have been updated (half), but it is still considerably underperforming the flat 3 prediction, and in a similar place to the random prediction. 

The obvious caveate is that there are far too few assets here to attempt to say these are representative statistics. 

The point is that, while it is possible to collect complete data on some of these embankments (lots were attempted and rejected), it is a time consuming process that leads to some still dubious data, and the predictions made with them still may contain inaccuracies. 



I intent to return to the problem of collecting more assets so that a larger sample size and more asset types can be used. 
The main caveates to the data collection process are:

    - the local filtering biases all data to one local region - this was necessary to scope the task of manually finding necessary data, but demonstrates that this is not doable (in this project scope) on a national scale
    - some assets were not possible to identify on satellite images and were rejected quickly
    - high quality virtical photography does not have wide coverage. In particular, this makes it hard to categorise embankment assets as bare or protected, since this requires a visible re enforced footing in the image. 
    - aerial images may not reveal protected footings of embankments if the protection is buried (possible on sand beaches) and cannot inform the internal materials of an asset (e.g. permiable vs clay core).
    - elevation profile used to categories wide and narrow crests on embankment were drawn perpendicular to the embankment by hand, meaning there could be a measurment error if the line is not at a perfect right angle, and then the crest width was judged subjectively, which introduces inconsistancy and bias.
    